# Lab | Langchain Evaluation

## Intro

Pick different sets of data and re-run this notebook. The point is for you to understand all steps involve and the many different ways one can and should evaluate LLM applications.

What did you learn? - Let's discuss that in class

## LangChain: Evaluation

### Outline:

* Example generation
* Manual evaluation (and debuging)
* LLM-assisted evaluation

> **This version runs on OpenAI's GPT Luna** (`gpt-6-luna`) with current LangChain. The original notebook used `RetrievalQA`, `QAGenerateChain`, `QAEvalChain` and `ragas==0.1.9`, which no longer exist in current LangChain, so each of them is rebuilt below with LCEL and Luna. The steps and the data are the same as the original. **Example 3** adds a different dataset, as the Intro asks.

Keys come from a `.env` file next to this notebook: `OPENAI_API_KEY=...` (and optionally `OPENAI_MODEL=gpt-6-luna`).

In [1]:
# %pip install -qU langchain-openai langchain-core langchain-community langchain-text-splitters python-dotenv pandas bs4

In [2]:
from dotenv import load_dotenv, find_dotenv
import os
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
assert OPENAI_API_KEY, "Add OPENAI_API_KEY to your .env file"

In [3]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# GPT Luna: OpenAI's fast, low-cost tier. It is a reasoning model, so we set reasoning_effort
# instead of temperature (Luna only accepts temperature when reasoning_effort="none").
LUNA_MODEL = os.getenv("OPENAI_MODEL", "gpt-6-luna")

def luna(effort: str = "low") -> ChatOpenAI:
    return ChatOpenAI(model=LUNA_MODEL, reasoning_effort=effort)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print(luna().invoke("Reply with the single word: ready").content)

ready


### Example 1

#### Create our QandA application

In [4]:
from langchain_community.document_loaders import CSVLoader, TextLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from typing import Literal

C:\Users\Roberto\AppData\Local\Temp\ipykernel_56992\135343278.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader, TextLoader


In [5]:
file = 'data/OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file, encoding="utf-8")
data = loader.load()
len(data)

1000

In [6]:
# Same idea as VectorstoreIndexCreator + DocArrayInMemorySearch: one document per catalog row,
# embedded and stored in an in-memory vector store (OpenAI embeddings instead of a local HF model).
index = InMemoryVectorStore.from_documents(data, embeddings)

In [7]:
# RetrievalQA ("stuff" chain) rebuilt with LCEL: retrieve docs, stuff them into the prompt, ask Luna.
# The output has the same keys as RetrievalQA: query, result, source_documents.
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "Use the following pieces of context to answer the user's question. "
               "If you don't know the answer, just say that you don't know, don't try to make up an answer.\n\n{context}"),
    ("human", "{question}"),
])

def make_qa_chain(retriever, llm, separator="\n\n"):
    answer = (
        (lambda x: {"context": separator.join(d.page_content for d in x["source_documents"]),
                    "question": x["query"]})
        | qa_prompt | llm | StrOutputParser()
    )
    return RunnableParallel(query=RunnablePassthrough(), source_documents=retriever) | RunnablePassthrough.assign(result=answer)

llm = luna()
qa = make_qa_chain(index.as_retriever(search_kwargs={"k": 4}), llm, separator="<<<<>>>>>")

#### Coming up with test datapoints

In [8]:
data[10]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 10}, page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\r\n\r\nSize & Fit\r\n- Pants are Favorite Fit: Sits lower on the waist.\r\n- Relaxed Fit: Our most generous fit sits farthest from the body.\r\n\r\nFabric & Care\r\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\r\n\r\nAdditional Features\r\n- Relaxed fit top with raglan sleeves and rounded hem.\r\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\r\n\r\nImported.")

In [9]:
data[11]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 11}, page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.')

#### Hard-coded examples

In [10]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

# Few-shot prompt with the hard-coded examples (same as the original LLMChain, now as LCEL)
prompt_template = PromptTemplate(
    input_variables=["query"],
    template="Examples:\n"
             "1. Query: Do the Cozy Comfort Pullover Set have side pockets?\n"
             "   Answer: Yes\n"
             "2. Query: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?\n"
             "   Answer: The DownTek collection\n"
             "Query: {query}\n"
             "Answer:"
)
llm_chain = prompt_template | llm | StrOutputParser()

# Without retrieval the model can only guess: compare this with the RAG answer below
query = "Is the Cozy Comfort Pullover Set available in different colors?"
print("No retrieval:", llm_chain.invoke({"query": query}))
print("With RAG:    ", qa.invoke(query)["result"])

No retrieval: Yes, it’s available in different colors.
With RAG:     Yes. The Cozy Comfort Pullover Set is listed as “Stripe,” but the provided information doesn’t specify which colors are available.


#### LLM-Generated examples

In [11]:
# QAGenerateChain rebuilt: Luna reads one catalog row and writes a question/answer pair (structured output)
class QAPair(BaseModel):
    query: str = Field(description="A specific question that can be answered from the document")
    answer: str = Field(description="The answer to the question, taken from the document")

gen_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a teacher coming up with questions to ask on a quiz. Given the following document, "
               "please generate a question and answer based on the document. The question must name the product "
               "so it can be answered without seeing the document."),
    ("human", "{doc}"),
])
example_gen_chain = gen_prompt | luna().with_structured_output(QAPair)

In [12]:
new_examples = example_gen_chain.batch([{"doc": t.page_content} for t in data[:5]])

In [13]:
new_examples[0]

QAPair(query="What features provide cushioning and support in the Women's Campside Oxfords?", answer='They have an EVA foam midsole for cushioning and support, along with a comfortable EVA innersole.')

In [14]:
data[0]

Document(metadata={'source': 'data/OutdoorClothingCatalog_1000.csv', 'row': 0}, page_content=": 0\nname: Women's Campside Oxfords\ndescription: This ultracomfortable lace-to-toe Oxford boasts a super-soft canvas, thick cushioning, and quality construction for a broken-in feel from the first time you put them on. \r\n\r\nSize & Fit: Order regular shoe size. For half sizes not offered, order up to next whole size. \r\n\r\nSpecs: Approx. weight: 1 lb.1 oz. per pair. \r\n\r\nConstruction: Soft canvas material for a broken-in feel and look. Comfortable EVA innersole with Cleansport NXT® antimicrobial odor control. Vintage hunt, fish and camping motif on innersole. Moderate arch contour of innersole. EVA foam midsole for cushioning and support. Chain-tread-inspired molded rubber outsole with modified chain-tread pattern. Imported. \r\n\r\nQuestions? Please contact us for any inquiries.")

In [15]:
d_flattened = [e.model_dump() for e in new_examples]
d_flattened

[{'query': "What features provide cushioning and support in the Women's Campside Oxfords?",
  'answer': 'They have an EVA foam midsole for cushioning and support, along with a comfortable EVA innersole.'},
 {'query': 'What materials are used to construct the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'It has 24 oz. polyester fabric made from 94% recycled materials and a rubber backing.'},
 {'query': "What sun protection does the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece provide?",
  'answer': 'Its UPF 50+ fabric blocks 98% of the sun’s harmful rays.'},
 {'query': 'What sun protection rating does the Refresh Swimwear V-Neck Tankini Contrasts provide?',
  'answer': 'It is UPF 50+, the highest rated sun protection possible.'},
 {'query': 'What features of the EcoFlex 3L Storm Pants help keep out rain and snow and make them easy to put on over boots?',
  'answer': 'Interior gaiters keep out rain and snow, and full side zips make the pants easy to put on and t

#### Combine examples

In [16]:
examples += d_flattened
len(examples)

7

In [17]:
examples[0]

{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
 'answer': 'Yes'}

In [18]:
qa.invoke(examples[0]["query"])["result"]

'Yes. The Cozy Comfort Pullover Set pants have side pockets.'

### Manual Evaluation - Fun part

In [19]:
from langchain_core.globals import set_debug
set_debug(True)

In [20]:
_ = qa.invoke(examples[0]["query"])

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<query,source_documents>] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<query,source_documents> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "input": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<query,source_documents> > chain:RunnablePassthrough] s] Exiting Chain run with output:
{
  "output": "Do the Cozy Comfort Pullover Set have side pockets?"
}
[chain/end] [chain:RunnableSequence > chain:RunnableParallel<query,source_documents>] s] Exiting Chain run with output:
[outputs]
[chain/start] [chain:RunnableSequence > chain:RunnableAssign<result>] Entering Chain run with input:
[inputs]

In [21]:
# Turn off the debug mode
set_debug(False)

### LLM assisted evaluation

In [22]:
examples

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'answer': 'Yes'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'answer': 'The DownTek collection'},
 {'query': "What features provide cushioning and support in the Women's Campside Oxfords?",
  'answer': 'They have an EVA foam midsole for cushioning and support, along with a comfortable EVA innersole.'},
 {'query': 'What materials are used to construct the Recycled Waterhog Dog Mat, Chevron Weave?',
  'answer': 'It has 24 oz. polyester fabric made from 94% recycled materials and a rubber backing.'},
 {'query': "What sun protection does the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece provide?",
  'answer': 'Its UPF 50+ fabric blocks 98% of the sun’s harmful rays.'},
 {'query': 'What sun protection rating does the Refresh Swimwear V-Neck Tankini Contrasts provide?',
  'answer': 'It is UPF 50+, the highest rated sun protection possible.'},
 {'query': 'What feature

In [23]:
predictions = qa.batch([e["query"] for e in examples])

In [24]:
[{"query": p["query"], "result": p["result"]} for p in predictions]

[{'query': 'Do the Cozy Comfort Pullover Set have side pockets?',
  'result': 'Yes. The Cozy Comfort Pullover Set’s pull-on pants have side pockets.'},
 {'query': 'What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?',
  'result': 'The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.'},
 {'query': "What features provide cushioning and support in the Women's Campside Oxfords?",
  'result': "The Women's Campside Oxfords have a comfortable EVA insole with a moderate arch contour, plus an EVA foam midsole for cushioning and support."},
 {'query': 'What materials are used to construct the Recycled Waterhog Dog Mat, Chevron Weave?',
  'result': 'It’s made with 24 oz. polyester fabric containing 94% recycled materials and has a rubber backing.'},
 {'query': "What sun protection does the Infant and Toddler Girls' Coastal Chill Swimsuit, Two-Piece provide?",
  'result': 'It has UPF 50+ fabric, which blocks 98% of the sun’s harmful rays.'},
 {'query':

In [25]:
# QAEvalChain rebuilt: Luna grades each predicted answer against the real answer
class Grade(BaseModel):
    reasoning: str = Field(description="Short explanation of the grade")
    grade: Literal["CORRECT", "INCORRECT"]

grade_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a teacher grading a quiz. You are given a question, the true answer, and a student answer. "
               "Grade the student answer based ONLY on its factual accuracy compared with the true answer. "
               "Ignore differences in punctuation and phrasing; extra information is fine as long as it does not "
               "contradict the true answer."),
    ("human", "QUESTION: {query}\nTRUE ANSWER: {answer}\nSTUDENT ANSWER: {result}"),
])
eval_chain = grade_prompt | luna().with_structured_output(Grade)

In [26]:
graded_outputs = eval_chain.batch([
    {"query": e["query"], "answer": e["answer"], "result": p["result"]}
    for e, p in zip(examples, predictions)
])
graded_outputs

[Grade(reasoning='The student correctly states that the set has side pockets.', grade='CORRECT'),
 Grade(reasoning='The student correctly identifies the DownTek collection.', grade='CORRECT'),
 Grade(reasoning='The answer correctly identifies the EVA foam midsole for cushioning and support and the comfortable EVA insole.', grade='CORRECT'),
 Grade(reasoning='The student accurately identifies the 24 oz. polyester fabric made from 94% recycled materials and the rubber backing.', grade='CORRECT'),
 Grade(reasoning='The student correctly states that the UPF 50+ fabric blocks 98% of the sun’s harmful rays.', grade='CORRECT'),
 Grade(reasoning='The student correctly states that the tankini provides UPF 50+ sun protection.', grade='CORRECT'),
 Grade(reasoning='The student identifies interior gaiters that keep out rain and snow and full side zips that make the pants easy to put on over boots.', grade='CORRECT')]

In [27]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query'])
    print("Real Answer: " + eg['answer'])
    print("Predicted Answer: " + predictions[i]['result'])
    print("Predicted Grade: " + graded_outputs[i].grade + " - " + graded_outputs[i].reasoning)
    print()

accuracy = sum(g.grade == "CORRECT" for g in graded_outputs) / len(graded_outputs)
print(f"Accuracy: {accuracy:.0%} ({len(graded_outputs)} examples)")

Example 0:
Question: Do the Cozy Comfort Pullover Set have side pockets?
Real Answer: Yes
Predicted Answer: Yes. The Cozy Comfort Pullover Set’s pull-on pants have side pockets.
Predicted Grade: CORRECT - The student correctly states that the set has side pockets.

Example 1:
Question: What collection is the Ultra-Lofty 850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Predicted Grade: CORRECT - The student correctly identifies the DownTek collection.

Example 2:
Question: What features provide cushioning and support in the Women's Campside Oxfords?
Real Answer: They have an EVA foam midsole for cushioning and support, along with a comfortable EVA innersole.
Predicted Answer: The Women's Campside Oxfords have a comfortable EVA insole with a moderate arch contour, plus an EVA foam midsole for cushioning and support.
Predicted Grade: CORRECT - The answer correctly ident

### Example 2
One can also easily evaluate your QA chains with the metrics offered in ragas

In [28]:
nyc_docs = TextLoader("data/nyc_text.txt", encoding="utf-8").load()
nyc_chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(nyc_docs)
nyc_index = InMemoryVectorStore.from_documents(nyc_chunks, embeddings)

qa_chain = make_qa_chain(nyc_index.as_retriever(search_kwargs={"k": 4}), luna())
print(len(nyc_chunks), "chunks")

209 chunks


In [29]:
# testing it out

question = "How did New York City get its name?"
result = qa_chain.invoke(question)
result["result"]

'New York City was named in 1664 for the Duke of York, later King James II. After England took control of the Dutch settlement of New Amsterdam, King Charles II granted the territory to his brother, the Duke, and the city was renamed New York.'

In [30]:
result

{'query': 'How did New York City get its name?',
 'source_documents': [Document(id='7d02c882-4bdd-42ce-afb7-420af1f58043', metadata={'source': 'data/nyc_text.txt'}, page_content="== Etymology ==\n\nIn 1664, New York was named in honor of the Duke of York, who would become King James II of England. James's elder brother, King Charles II, appointed the Duke as proprietor of the former territory of New Netherland, including the city of New Amsterdam, when England seized it from Dutch control.\n\n\n== History =="),
  Document(id='dc10c14e-c209-45e9-a635-e25ce8bf82cb', metadata={'source': 'data/nyc_text.txt'}, page_content="In the pre-Columbian era, the area of present-day New York City was inhabited by Algonquian Native Americans, including the Lenape. Their homeland, known as Lenapehoking, included the present-day areas of Staten Island, Manhattan, the Bronx, the western portion of Long Island (including the areas that would later become the boroughs of Brooklyn and Queens), and the Lower

Now in order to evaluate the qa system we generated a few relevant questions. We've generated a few question for you but feel free to add any you want.

In [31]:
eval_questions = [
    "What is the population of New York City as of 2020?",
    "Which borough of New York City has the highest population?",
    "What is the economic significance of New York City?",
    "How did New York City get its name?",
    "What is the significance of the Statue of Liberty in New York City?",
]

eval_answers = [
    "8,804,190",
    "Brooklyn",
    "New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter.",
    "New York City got its name when it came under British control in 1664. King Charles II of England granted the lands to his brother, the Duke of York, who named the city New York in his own honor.",
    "The Statue of Liberty in New York City holds great significance as a symbol of the United States and its ideals of liberty and peace. It greeted millions of immigrants who arrived in the U.S. by ship in the late 19th and early 20th centuries, representing hope and freedom for those seeking a better life. It has since become an iconic landmark and a global symbol of cultural diversity and freedom.",
]

examples = [
    {"query": q, "ground_truths": [eval_answers[i]]}
    for i, q in enumerate(eval_questions)
]

In [32]:
examples

[{'query': 'What is the population of New York City as of 2020?',
  'ground_truths': ['8,804,190']},
 {'query': 'Which borough of New York City has the highest population?',
  'ground_truths': ['Brooklyn']},
 {'query': 'What is the economic significance of New York City?',
  'ground_truths': ["New York City's economic significance is vast, as it serves as the global financial capital, housing Wall Street and major financial institutions. Its diverse economy spans technology, media, healthcare, education, and more, making it resilient to economic fluctuations. NYC is a hub for international business, attracting global companies, and boasts a large, skilled labor force. Its real estate market, tourism, cultural industries, and educational institutions further fuel its economic prowess. The city's transportation network and global influence amplify its impact on the world stage, solidifying its status as a vital economic player and cultural epicenter."]},
 {'query': 'How did New York City

#### Introducing RagasEvaluatorChain
The original notebook used `ragas==0.1.9`'s `EvaluatorChain`, which doesn't work with current LangChain. Below, the four metrics it used are computed with **the same definitions as Ragas**, using Luna as the judge:

| Metric | Question it answers | How it's computed (Ragas definition) |
|---|---|---|
| **faithfulness** | Is the answer supported by the retrieved context? | split the answer into claims → share of claims the context supports |
| **answer_relevancy** | Does the answer address the question? | generate questions from the answer → mean cosine similarity to the real question |
| **context_relevancy** | Is the retrieved context focused on the question? | share of context sentences needed to answer the question |
| **context_recall** | Does the context contain the ground truth? | share of ground-truth sentences attributable to the context |

All scores are between 0 and 1, higher is better.

In [33]:
result = qa_chain.invoke(eval_questions[1])
result["result"]

'Brooklyn has the highest population of New York City’s five boroughs.'

In [34]:
key_mapping = {
    "query": "question",
    "result": "answer",
    "source_documents": "contexts"
}

result_updated = {}
for old_key, new_key in key_mapping.items():
    if old_key in result:
        result_updated[new_key] = result[old_key]

In [35]:
result_updated

{'question': 'Which borough of New York City has the highest population?',
 'answer': 'Brooklyn has the highest population of New York City’s five boroughs.',
 'contexts': [Document(id='0eb51c3e-339b-4712-8026-7574c889055e', metadata={'source': 'data/nyc_text.txt'}, page_content="New York is also home to the highest Jewish population of any city in the world, numbering 1.6 million in 2022, more than Tel Aviv and Jerusalem combined. In the borough of Brooklyn, an estimated 1 in 4 residents is Jewish. The city's Jewish communities are derived from many diverse sects, predominantly from around the Middle East and Eastern Europe, and including a rapidly growing Orthodox Jewish population, also the largest outside Israel.The metropolitan area is also home to 20% of the nation's Indian Americans and at least 20 Little India enclaves, and 15% of all Korean Americans and four Koreatowns; the largest Asian Indian population in the Western Hemisphere; the largest Russian American, Italian Americ

In [36]:
import re
import numpy as np

judge = luna()

def _contexts(docs):
    return "\n\n".join(d.page_content if hasattr(d, "page_content") else str(d) for d in docs)

def _sentences(text):
    return [s for s in re.split(r"(?<=[.!?])\s+|\n+", text) if len(s.strip()) > 3]

class Claims(BaseModel):
    claims: list[str] = Field(description="Atomic factual statements made in the answer")
class Verdicts(BaseModel):
    supported: list[bool] = Field(description="One verdict per statement, in order: true if the context supports it")
class GenQuestions(BaseModel):
    questions: list[str] = Field(description="3 questions that the answer would be a good answer to")
class RelevantSentences(BaseModel):
    sentences: list[str] = Field(description="Sentences copied from the context that are needed to answer the question")

def faithfulness(question, answer, contexts):
    claims = judge.with_structured_output(Claims).invoke(
        f"Break this answer into short, atomic factual statements.\nQuestion: {question}\nAnswer: {answer}").claims
    if not claims:
        return 0.0
    verdicts = judge.with_structured_output(Verdicts).invoke(
        "For each statement, say whether it can be directly inferred from the context.\n"
        f"CONTEXT:\n{_contexts(contexts)}\n\nSTATEMENTS:\n" + "\n".join(f"{i+1}. {c}" for i, c in enumerate(claims))).supported
    return float(np.mean(verdicts[:len(claims)])) if verdicts else 0.0

def answer_relevancy(question, answer, contexts=None):
    qs = judge.with_structured_output(GenQuestions).invoke(
        f"Write 3 different questions that this answer would be a good answer to.\nAnswer: {answer}").questions
    if not qs:
        return 0.0
    q_vec = np.array(embeddings.embed_query(question))
    g_vecs = np.array(embeddings.embed_documents(qs))
    sims = g_vecs @ q_vec / (np.linalg.norm(g_vecs, axis=1) * np.linalg.norm(q_vec))
    return float(sims.mean())

def context_relevancy(question, answer, contexts):
    ctx = _contexts(contexts)
    total = len(_sentences(ctx))
    picked = judge.with_structured_output(RelevantSentences).invoke(
        "Copy, word for word, only the sentences from the context that are needed to answer the question. "
        f"Return an empty list if none are.\nQuestion: {question}\nCONTEXT:\n{ctx}").sentences
    return min(1.0, len(picked) / total) if total else 0.0

def context_recall(question, answer, contexts, ground_truth):
    gt_sentences = _sentences(ground_truth) or [ground_truth]
    verdicts = judge.with_structured_output(Verdicts).invoke(
        "For each statement from the ground-truth answer, say whether it can be attributed to the context.\n"
        f"CONTEXT:\n{_contexts(contexts)}\n\nSTATEMENTS:\n" + "\n".join(f"{i+1}. {s}" for i, s in enumerate(gt_sentences))).supported
    return float(np.mean(verdicts[:len(gt_sentences)])) if verdicts else 0.0

1. `__call__()`

Directly run the evaluation chain with the results from the QA chain. Do note that metrics like context_relevancy and faithfulness require the `source_documents` to be present.

In [37]:
# Recheck the result that we are going to validate.
result

{'query': 'Which borough of New York City has the highest population?',
 'source_documents': [Document(id='0eb51c3e-339b-4712-8026-7574c889055e', metadata={'source': 'data/nyc_text.txt'}, page_content="New York is also home to the highest Jewish population of any city in the world, numbering 1.6 million in 2022, more than Tel Aviv and Jerusalem combined. In the borough of Brooklyn, an estimated 1 in 4 residents is Jewish. The city's Jewish communities are derived from many diverse sects, predominantly from around the Middle East and Eastern Europe, and including a rapidly growing Orthodox Jewish population, also the largest outside Israel.The metropolitan area is also home to 20% of the nation's Indian Americans and at least 20 Little India enclaves, and 15% of all Korean Americans and four Koreatowns; the largest Asian Indian population in the Western Hemisphere; the largest Russian American, Italian American, and African American populations; the largest Dominican American, Puerto Ri

**Faithfulness**

In [38]:
faithfulness(result_updated["question"], result_updated["answer"], result_updated["contexts"])

0.0

High faithfulness_score means that there are exact consistency between the source documents and the answer.

You can check lower faithfulness scores by changing the result (answer from LLM) or source_documents to something else.

In [39]:
fake_result = result_updated.copy()
fake_result["answer"] = "we are the champions"
faithfulness(fake_result["question"], fake_result["answer"], fake_result["contexts"])

1.0

**Context Recall**

In [40]:
ground_truth = examples[1]["ground_truths"][0]
context_recall(result_updated["question"], result_updated["answer"], result_updated["contexts"], ground_truth)

1.0

High context_recall_score means that the ground truth is present in the source documents.

You can check lower context recall scores by changing the source_documents to something else.

In [41]:
from langchain_core.documents import Document
fake_result = result_updated.copy()
fake_result["contexts"] = [Document(page_content="I love christmas")]
context_recall(fake_result["question"], fake_result["answer"], fake_result["contexts"], ground_truth)

0.0

**Answer relevancy and context relevancy**

In [42]:
print("answer_relevancy :", round(answer_relevancy(result_updated["question"], result_updated["answer"]), 3))
print("context_relevancy:", round(context_relevancy(result_updated["question"], result_updated["answer"], result_updated["contexts"]), 3))

answer_relevancy : 0.796
context_relevancy: 0.0


2. `evaluate()`

Evaluate a list of inputs/queries and the outputs/predictions from the QA chain.

In [43]:
import pandas as pd

def evaluate_rag(chain, examples):
    """Run the chain on every example and compute the four Ragas-style metrics."""
    predictions = chain.batch([e["query"] for e in examples])
    rows = []
    for e, p in zip(examples, predictions):
        ctx = p["source_documents"]
        rows.append({
            "question": e["query"],
            "answer": p["result"],
            "faithfulness": faithfulness(e["query"], p["result"], ctx),
            "answer_relevancy": answer_relevancy(e["query"], p["result"]),
            "context_relevancy": context_relevancy(e["query"], p["result"], ctx),
            "context_recall": context_recall(e["query"], p["result"], ctx, e["ground_truths"][0]),
        })
    return pd.DataFrame(rows)

# run the queries as a batch and evaluate
print("evaluating...")
nyc_scores = evaluate_rag(qa_chain, examples)
nyc_scores.round(2)

evaluating...


,question,answer,faithfulness,answer_relevancy,context_relevancy,context_recall
0,What is the population of New York City as of ...,"New York City’s population was 8,804,190 in 2020.",1.0,0.88,0.04,1.0
1,Which borough of New York City has the highest...,Brooklyn has the highest population of New Yor...,0.0,0.86,0.00,1.0
2,What is the economic significance of New York ...,"New York City is a major global economic hub, ...",1.0,0.70,0.12,0.2
3,How did New York City get its name?,"In 1664, England took control of New Amsterdam...",1.0,0.53,0.09,0.5
4,What is the significance of the Statue of Libe...,The Statue of Liberty is a symbol of liberty a...,1.0,0.69,0.04,0.5


In [44]:
nyc_scores[["faithfulness", "answer_relevancy", "context_relevancy", "context_recall"]].mean().round(3)

faithfulness         0.800
answer_relevancy     0.732
context_relevancy    0.058
context_recall       0.640
dtype: float64

### Example 3 - a different dataset
The Intro asks to pick different data and re-run the evaluation. Here the same pipeline and the same four metrics run on Lilian Weng's blog post on **prompt engineering**, with my own questions and reference answers.

In [45]:
from langchain_community.document_loaders import WebBaseLoader

pe_docs = WebBaseLoader("https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/").load()
pe_chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100).split_documents(pe_docs)
pe_chain = make_qa_chain(InMemoryVectorStore.from_documents(pe_chunks, embeddings).as_retriever(search_kwargs={"k": 4}), luna())

pe_examples = [
    {"query": "What is prompt engineering?",
     "ground_truths": ["Prompt engineering, also known as in-context prompting, refers to methods for how to communicate with an LLM to steer its behavior for desired outcomes without updating the model weights."]},
    {"query": "What biases can arise with few-shot prompting?",
     "ground_truths": ["Majority label bias, recency bias, and common token bias."]},
    {"query": "What is self-consistency sampling?",
     "ground_truths": ["Sampling multiple outputs with temperature > 0 and then selecting the best one out of these candidates, often by majority vote."]},
    {"query": "What are the two main types of chain-of-thought prompting?",
     "ground_truths": ["Few-shot CoT, which prompts the model with demonstrations that include reasoning chains, and zero-shot CoT, which uses a phrase like 'Let's think step by step'."]},
    {"query": "What is the capital of France?",   # out-of-scope question: the model should say it doesn't know
     "ground_truths": ["The document does not contain this information."]},
]

print("evaluating...")
pe_scores = evaluate_rag(pe_chain, pe_examples)
pe_scores.round(2)

USER_AGENT environment variable not set, consider setting it to identify your requests.


evaluating...


,question,answer,faithfulness,answer_relevancy,context_relevancy,context_recall
0,What is prompt engineering?,Prompt engineering is the practice of designin...,1.0,0.61,0.02,1.0
1,What biases can arise with few-shot prompting?,Few-shot prompting can exhibit three biases:\n...,1.0,0.69,0.02,1.0
2,What is self-consistency sampling?,Self-consistency sampling generates multiple c...,1.0,0.59,0.05,1.0
3,What are the two main types of chain-of-though...,The two main types are **zero-shot chain-of-th...,0.0,0.81,0.00,0.0
4,What is the capital of France?,Paris.,0.0,0.88,0.00,1.0


In [46]:
summary = pd.DataFrame({
    "NYC text (Example 2)": nyc_scores[["faithfulness", "answer_relevancy", "context_relevancy", "context_recall"]].mean(),
    "Prompt engineering post (Example 3)": pe_scores[["faithfulness", "answer_relevancy", "context_relevancy", "context_recall"]].mean(),
}).round(3)
summary

,NYC text (Example 2),Prompt engineering post (Example 3)
faithfulness,0.800,0.600
answer_relevancy,0.732,0.720
context_relevancy,0.058,0.017
context_recall,0.640,0.800


### What did you learn?
- **Evaluation needs test data first.** Hand-written examples are precise but slow to write; LLM-generated examples (Example 1) scale easily but need a quick review, since the generator can write vague questions.
- **Manual evaluation (debug mode) shows *why* an answer is wrong.** You can see exactly which documents were retrieved and what prompt the model received, which separates retrieval problems from generation problems.
- **LLM-assisted grading compares meaning, not strings.** "Yes" and "Yes, it has side pockets" are both CORRECT; exact-match checks would fail them. But the judge is itself an LLM, so its verdicts and explanations are worth spot-checking.
- **Different metrics catch different failures.** *Faithfulness* catches hallucination (the fake "we are the champions" answer scores near 0), *context recall* catches retrieval misses (the fake "I love christmas" context scores near 0), *answer relevancy* catches off-topic answers and *context relevancy* shows how much retrieved text is noise.
- **Results depend on the data.** Running the same pipeline on a second dataset (Example 3) shows the scores are not a fixed property of the model: chunking, retrieval and the kind of questions all change them, so every new dataset needs its own evaluation.